<a href="https://colab.research.google.com/github/shanusushmita/CS4973-Applied-Multilingual-Systems/blob/main/Assignment5_Bias_Analysis_Scaffold.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment 5 — Bias Analysis Notebook (Scaffold)

This notebook is a scaffold to help your group perform the **bias analyses** required for Assignment 5. It **assumes** your synthetic dataset is stored as JSONL or JSON with a conversation structure. Edit the data-loading cell to point to your dataset file.

The notebook includes helper functions for:
- Loading JSON / JSONL conversation data
- Heuristic language detection for English / Spanish (Spain) / Mandarin / Hindi
- Counting tokens per language, switches, intersentential vs intrasentential switches
- Simple visualizations and exports for inclusion in your report

### Notes
- This scaffold uses string/script heuristics for Mandarin/Hindi detection and optionally `langdetect` if available. If you want to use a more robust detector, install and run it locally in your environment and adapt the `detect_language()` function.
- Edit the field names used for parsing if your JSON structure differs.

-----
## How to use
1. Upload your dataset into the environment (or change the path in the data-load cell).
2. Run cells top-to-bottom, inspecting outputs and saving visualizations.
3. Use the `export_report()` helper to save results (CSV/JSON) for your Assignment 5 submission.


In [ ]:
# Imports
import json
from collections import Counter, defaultdict
import re
import math
import os
import pandas as pd
import matplotlib.pyplot as plt

try:
    from langdetect import detect
    LANGDETECT_AVAILABLE = True
except Exception:
    LANGDETECT_AVAILABLE = False

print('langdetect available:', LANGDETECT_AVAILABLE)


In [ ]:
# -----------------------------
# Data loading helper (edit path as needed)
DATA_PATH = '/mnt/data/your_generated_dataset.jsonl'  # <-- change this to your file

def load_jsonl(path):
    data = []
    if not os.path.exists(path):
        print('File not found:', path)
        return data
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError:
                # try parse as full json
                pass
    return data

def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_data(path):
    if path.endswith('.jsonl') or path.endswith('.jl'):
        return load_jsonl(path)
    elif path.endswith('.json'):
        return load_json(path)
    else:
        print('Unknown extension. Try .jsonl or .json')
        return []

# Load data
raw = load_data(DATA_PATH)
print('Loaded', len(raw), 'records')


In [ ]:
# -----------------------------
# Heuristic language detection
SPANISH_WORDS = set(['el','la','de','que','y','en','un','una','pero','por','para','con','como','no','si','porque','yo','tu','usted','vale','tio','guay','ordenador'])
ENGLISH_WORDS = set(['the','and','is','in','to','of','that','it','for','on','with','as','this','we','you','software','engineer'])

devanagari_pattern = re.compile(r'[\u0900-\u097F]')
chinese_pattern = re.compile(r'[\u4e00-\u9fff]')

def simple_lang_by_script_or_lexicon(text):
    # quick script checks
    if chinese_pattern.search(text):
        return 'zh'  # Mandarin (hanzi present)
    if devanagari_pattern.search(text):
        return 'hi'  # Hindi (Devanagari present)
    # token-based heuristic
    tokens = re.findall(r"\w+", text.lower())
    if not tokens:
        return 'und'
    en_score = sum(1 for t in tokens if t in ENGLISH_WORDS)
    es_score = sum(1 for t in tokens if t in SPANISH_WORDS)
    # tie-breaking
    if en_score == 0 and es_score == 0:
        # fallback to langdetect if available
        if LANGDETECT_AVAILABLE:
            try:
                l = detect(text)
                return l
            except Exception:
                return 'und'
        return 'und'
    return 'en' if en_score >= es_score else 'es'

def detect_language(text):
    # Wrapper: try langdetect, then heuristic
    if not text or not text.strip():
        return 'und'
    if LANGDETECT_AVAILABLE:
        try:
            return detect(text)
        except Exception:
            return simple_lang_by_script_or_lexicon(text)
    else:
        return simple_lang_by_script_or_lexicon(text)

# quick tests
tests = ['Hello, how are you?', '¿Qué tal estás?', '你好，今天怎么样？', 'क्या हाल है']
for t in tests:
    print(t, '->', detect_language(t))


In [ ]:
# -----------------------------
# Parsing a standard conversation record and computing switch stats
def parse_conversation(rec):
    """
    Expected a record representing a conversation. Common conventions:
    - rec could be a dict with key 'conversation' which is a list of turns
    - each turn is a dict with fields like 'speaker' and 'text'
    Adapt this function if your structure differs.
    """
    conv = rec
    if isinstance(rec, dict) and 'conversation' in rec:
        conv = rec['conversation']
    # Normalize to list of (speaker, text)
    turns = []
    for t in conv:
        if isinstance(t, dict):
            speaker = t.get('speaker', t.get('role', 'S'))
            text = t.get('text') or t.get('utterance') or ''
        else:
            # assume tuple/list [speaker,text] or plain string
            if isinstance(t, (list,tuple)) and len(t) >= 2:
                speaker, text = t[0], t[1]
            else:
                speaker, text = 'S', str(t)
        turns.append({'speaker': speaker, 'text': text})
    return turns

def analyze_conversation_switches(turns):
    # returns summary: token counts per lang, switches counts, inter/intra counts
    token_counts = Counter()
    lang_by_turn = []
    intra_switch_count = 0
    inter_switch_count = 0
    for turn in turns:
        text = turn['text']
        # naive sentence split by punctuation
        sentences = re.split(r'[\\.!?]+', text)
        sentence_langs = [detect_language(s) for s in sentences if s.strip()]
        # token-level language approx: detect on whole turn
        turn_lang = detect_language(text)
        lang_by_turn.append(turn_lang)
        # token count
        token_counts[turn_lang] += len(re.findall(r"\w+", text))
        # intra-sentential: if one sentence contains mixture, approximated by checking scripts
        for s in sentences:
            if not s.strip():
                continue
            # crude check: presence of both chinese/devanagari and ascii words or English/Spanish tokens
            langs_in_s = set()
            if chinese_pattern.search(s):
                langs_in_s.add('zh')
            if devanagari_pattern.search(s):
                langs_in_s.add('hi')
            tokens = re.findall(r"\w+", s.lower())
            en_score = sum(1 for t in tokens if t in ENGLISH_WORDS)
            es_score = sum(1 for t in tokens if t in SPANISH_WORDS)
            if en_score > 0:
                langs_in_s.add('en')
            if es_score > 0:
                langs_in_s.add('es')
            if len(langs_in_s) >= 2:
                intra_switch_count += 1
    # inter-sentential switches: adjacent turn languages differ
    for a,b in zip(lang_by_turn, lang_by_turn[1:]):
        if a != b and a != 'und' and b != 'und':
            inter_switch_count += 1
    return {
        'token_counts': dict(token_counts),
        'intra_switches': intra_switch_count,
        'inter_switches': inter_switch_count,
        'turn_langs': lang_by_turn
    }


In [ ]:
# -----------------------------
# Batch analysis over dataset
def analyze_dataset(records):
    results = []
    agg = defaultdict(int)
    for i,rec in enumerate(records):
        turns = parse_conversation(rec)
        out = analyze_conversation_switches(turns)
        out['id'] = i
        results.append(out)
        agg['total_intra'] += out['intra_switches']
        agg['total_inter'] += out['inter_switches']
        for k,v in out['token_counts'].items():
            agg[f'tokens_{k}'] += v
    return results, agg

# Run if data loaded
if raw:
    results, agg = analyze_dataset(raw)
    print('Conversations analyzed:', len(results))
    print('Aggregate intra switches:', agg['total_intra'])
    print('Aggregate inter switches:', agg['total_inter'])
    print('Token aggregates:', {k:v for k,v in agg.items() if k.startswith('tokens_')})
else:
    print('No data loaded. Edit DATA_PATH to point to your file and re-run this cell.')


In [ ]:
# -----------------------------
# Visualization helpers and export
def plot_token_distribution(agg):
    token_items = [(k.replace('tokens_',''),v) for k,v in agg.items() if k.startswith('tokens_')]
    if not token_items:
        print('No token data to plot')
        return
    df = pd.DataFrame(token_items, columns=['lang','tokens'])
    df.set_index('lang', inplace=True)
    df.plot.bar(rot=0)
    plt.title('Token counts per language')
    plt.ylabel('tokens')
    plt.show()

def export_summary(results, agg, out_path='/mnt/data/bias_analysis_summary.json'):
    summary = {'agg': agg, 'details': results}
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)
    print('Exported summary to', out_path)

if raw:
    plot_token_distribution(agg)


----
## Next steps and custom analyses

Use the helper functions above as a starting point. Recommended custom analyses (add cells below):
- Persona × language token heatmap
- Topic × language association table (if your records contain topic labels)
- Sentiment vs language analysis (use any sentiment tool you prefer locally)
- Manual sample review: pick N=30 examples per persona and inspect for stereotypical language

When finished, export your summary with `export_summary(results, agg)` and include the exported JSON plus visualizations in your Assignment 5 report.
